### Loading the datasets 

In [167]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error, mean_absolute_error
from numpy.linalg import svd

In [168]:
# Load all review files
files = [
    "Datasets/reviews_0-250.csv",
    "Datasets/reviews_250-500.csv",
    "Datasets/reviews_500-750.csv",
    "Datasets/reviews_750-1250.csv",
    "Datasets/reviews_1250-end.csv"
]

df_reviews = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_90569/3120905185.py:10: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_90569/3120905185.py:10: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_90569/3120905185.py:10: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


In [169]:
print(df_reviews.shape)

(1094411, 19)


In [170]:
df_reviews.columns

Index(['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name', 'brand_name', 'price_usd'],
      dtype='object')

one user
one product
one rating/review
maybe review text, helpfulness, submission info, skin type, etc.

So this table answers:

Who used which product?
Did they like it?
What did they say about it?

In [171]:
df_reviews = df_reviews.drop(columns=["Unnamed: 0"], errors="ignore")

In [172]:
print(df_reviews.isnull().sum().sort_values(ascending=False))

helpfulness                 561592
review_title                310654
hair_color                  226768
eye_color                   209628
skin_tone                   170539
is_recommended              167988
skin_type                   111557
review_text                   1444
brand_name                       0
product_name                     0
product_id                       0
author_id                        0
rating                           0
submission_time                  0
total_pos_feedback_count         0
total_neg_feedback_count         0
total_feedback_count             0
price_usd                        0
dtype: int64


In [173]:
df_products = pd.read_csv("Datasets/product_info.csv")

print(df_products.shape)

(8494, 27)


In [174]:
df_products.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='object')

Each row is one product, with things like:

product name
brand
category
ingredients
price
maybe size / highlights / description

So this table answers:

What kind of product is this?
What ingredients does it contain?
Is it a serum, cleanser, moisturizer, sunscreen, etc.?

df_reviews = the behavior table,   df_products = the product knowledge table

Merge both the datasets (who used what + what that product actually is)

In [175]:
df_products_original = df_products

In [176]:
print(df_products.isnull().sum().sort_values(ascending=False))

sale_price_usd        8224
value_price_usd       8043
variation_desc        7244
child_max_price       5740
child_min_price       5740
highlights            2207
size                  1631
variation_value       1598
variation_type        1444
tertiary_category      990
ingredients            945
rating                 278
reviews                278
secondary_category       8
sephora_exclusive        0
brand_id                 0
child_count              0
primary_category         0
new                      0
out_of_stock             0
online_only              0
limited_edition          0
brand_name               0
product_name             0
price_usd                0
loves_count              0
product_id               0
dtype: int64


Note: df_reviews["rating"] = user-specific rating
df_products["rating"] = average product rating across all users

In [177]:
# Keep only skincare products
df_products = df_products[df_products["primary_category"] == "Skincare"].copy()

print("Filtered df_products shape:", df_products.shape)
print(df_products["primary_category"].value_counts(dropna=False))

Filtered df_products shape: (2420, 27)
primary_category
Skincare    2420
Name: count, dtype: int64


In [178]:
df_products = df_products.rename(columns={"rating": "avg_product_rating",
                                          "reviews": "num_reviews"})

In [179]:
print("Original df_products shape:", df_products_original.shape)
print("Filtered skincare df_products shape:", df_products.shape)

Original df_products shape: (8494, 27)
Filtered skincare df_products shape: (2420, 27)


In [180]:
print(df_products["primary_category"].value_counts(dropna=False))

primary_category
Skincare    2420
Name: count, dtype: int64


In [181]:
all_review_products = set(df_reviews["product_id"].unique())
skincare_products = set(df_products["product_id"].unique())   # filtered table

non_skincare_review_products = all_review_products - skincare_products

print("Unique product_ids in reviews:", len(all_review_products))
print("Unique skincare product_ids:", len(skincare_products))
print("Review product_ids not in skincare set:", len(non_skincare_review_products))

Unique product_ids in reviews: 2351
Unique skincare product_ids: 2420
Review product_ids not in skincare set: 0


### Merging the review and product dataset 

In [182]:
cols_to_take = [
    "product_id", "product_name", "brand_name", "ingredients", "price_usd",
    "highlights", "primary_category", "secondary_category", "tertiary_category",
    "avg_product_rating", "num_reviews", "loves_count",
    "sale_price_usd", "value_price_usd"
]

df_merged = df_reviews.merge(
    df_products[cols_to_take],
    on="product_id",
    how="inner"
)

In [183]:
print("Merged skincare-only dataset shape:", df_merged.shape)

Merged skincare-only dataset shape: (1094411, 31)


In [184]:
df_merged.columns

Index(['author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name_x', 'brand_name_x', 'price_usd_x',
       'product_name_y', 'brand_name_y', 'ingredients', 'price_usd_y',
       'highlights', 'primary_category', 'secondary_category',
       'tertiary_category', 'avg_product_rating', 'num_reviews', 'loves_count',
       'sale_price_usd', 'value_price_usd'],
      dtype='object')

We have 2420 unique skincare products and 1,094,411 reviews for different products.

From the merged dataset, we have different kinds of information: What kind of products does a particular user tend to like, What kind of product is this, and why might it suit someone. There may be many usersbut there are only 2420 products so each user has only reviewed a tiny fraction of all products. Most user-product cells are missing. Can we intelligently fill in the missing preferences. 

In [185]:
print("Unique products in df_reviews:", df_reviews["product_id"].nunique())
print("Unique products in merged df:", df_merged["product_id"].nunique())

Unique products in df_reviews: 2351
Unique products in merged df: 2351


In [186]:
n_users = df_merged["author_id"].nunique()
n_products = df_merged["product_id"].nunique()
n_interactions = len(df_merged)

reviews_per_user = df_merged.groupby("author_id").size()
reviews_per_product = df_merged.groupby("product_id").size()

print("Unique users:", n_users)
print("Unique products:", n_products)
print("Interactions:", n_interactions)
print("Avg reviews per user:", reviews_per_user.mean())
print("Avg reviews per product:", reviews_per_product.mean())

Unique users: 578653
Unique products: 2351
Interactions: 1094411
Avg reviews per user: 1.8913079168344413
Avg reviews per product: 465.50871969374737


In [187]:
print(df_merged["rating"].value_counts().sort_index())

rating
1     61223
2     53032
3     81816
4    199389
5    698951
Name: count, dtype: int64


In [188]:
important_cols = [
    "author_id", "product_id", "rating", "review_text",
    "skin_type", "ingredients", "primary_category"
]

print(df_merged[important_cols].isnull().mean().sort_values(ascending=False))

skin_type           0.101933
ingredients         0.020125
review_text         0.001319
author_id           0.000000
product_id          0.000000
rating              0.000000
primary_category    0.000000
dtype: float64


In [189]:
df_model = df_merged[
    [
        "author_id", "product_id", "rating", "is_recommended",
        "review_text", "skin_type", "skin_tone", "eye_color", "hair_color",
        "product_name_y", "brand_name_y", "price_usd_y", "ingredients", "highlights",
        "primary_category", "secondary_category", "tertiary_category",
        "avg_product_rating", "num_reviews", "loves_count"
    ]
].copy()

In [190]:
# Build Phase A interaction table
df_phaseA = df_merged[["author_id", "product_id", "rating"]].copy()
df_phaseA = df_phaseA.dropna(subset=["author_id", "product_id", "rating"])

duplicate_pairs = df_phaseA.duplicated(subset=["author_id", "product_id"]).sum()
print("Duplicate user-product pairs:", duplicate_pairs)

df_phaseA_clean = (
    df_phaseA
    .groupby(["author_id", "product_id"], as_index=False)["rating"]
    .mean()
)

print("Shape after collapsing duplicates:", df_phaseA_clean.shape)

Duplicate user-product pairs: 5520
Shape after collapsing duplicates: (1088891, 3)


It is too sparse + too large so it is hard to learn patterns

Since we have 99.9% sparse cells, we filter the table with users >= 5 reviews and products >= 10 reviews.

We removed users who barely reviewed anything and products with almost no data.

Before filtering, User A rated 1 product so no pattern observed. After filtering, User A rated 10 products so it will be easier to learn patterns.

Our goal here is to predict the missing values. Matrix factorisation will learn similar users and similar products nd fill the missing values.

### Filter active users and products

In [191]:
user_counts = df_phaseA_clean["author_id"].value_counts()
product_counts = df_phaseA_clean["product_id"].value_counts()

active_users = user_counts[user_counts >= 10].index
active_products = product_counts[product_counts >= 20].index

df_phaseA_filtered = df_phaseA_clean[
    df_phaseA_clean["author_id"].isin(active_users) &
    df_phaseA_clean["product_id"].isin(active_products)
].copy()

print("Filtered interaction table shape:", df_phaseA_filtered.shape)
df_phaseA_filtered.head()

Filtered interaction table shape: (133720, 3)


,author_id,product_id,rating
93,2760276,P232915,5.0
94,2760276,P309409,4.0
95,2760276,P386762,3.0
97,2760276,P420652,5.0
98,2760276,P427416,2.0


### Build grid for user product rating

In [192]:
A = df_phaseA_filtered.pivot(
    index="author_id",
    columns="product_id",
    values="rating"
)

print("Filtered matrix shape:", A.shape)
A.head()

Filtered matrix shape: (7029, 1752)


product_id,P107306,P114902,P12045,P122651,P122661,P122718,P122727,P122762,P122767,P122774,...,P54509,P6028,P7365,P7880,P91627362,P94421,P94812,P9939,P9940,P9941
author_id,,,,,,,,,,,,,,,,,,,,,
2760276,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN
11777122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
967124371,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN
985770407,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
989697609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [193]:
n_users, n_products = A.shape

total_cells = n_users * n_products
observed_cells = A.notna().sum().sum()
missing_cells = A.isna().sum().sum()
missing_pct = (missing_cells / total_cells) * 100

print("Number of users:", n_users)
print("Number of products:", n_products)
print("Total cells:", total_cells)
print("Observed cells:", observed_cells)
print("Missing cells:", missing_cells)
print("Percent missing:", round(missing_pct, 4), "%")

Number of users: 7029
Number of products: 1752
Total cells: 12314808
Observed cells: 133720
Missing cells: 12181088
Percent missing: 98.9142 %


In [194]:
observed_mask = A.notna()

In [195]:
global_mean = df_phaseA_filtered["rating"].mean()

A_filled = A.fillna(A.mean()).fillna(global_mean)

A_filled.head()

product_id,P107306,P114902,P12045,P122651,P122661,P122718,P122727,P122762,P122767,P122774,...,P54509,P6028,P7365,P7880,P91627362,P94421,P94812,P9939,P9940,P9941
author_id,,,,,,,,,,,,,,,,,,,,,
2760276,3.826087,4.404762,4.47619,3.875,3.994253,4.333333,3.8,4.0,3.428571,3.862319,...,4.636364,4.071429,3.777778,2.000000,4.266667,4.117647,4.444444,4.392857,4.21519,4.423077
11777122,3.826087,4.404762,4.47619,3.875,3.994253,4.333333,3.8,4.0,3.428571,3.862319,...,4.636364,4.071429,3.777778,3.681818,4.266667,4.117647,4.444444,4.392857,4.21519,4.423077
967124371,3.826087,4.404762,4.47619,3.875,3.994253,4.333333,3.8,4.0,3.428571,3.862319,...,4.636364,4.071429,3.777778,4.000000,4.266667,4.117647,4.444444,4.392857,4.21519,4.423077
985770407,3.826087,4.404762,4.47619,3.875,3.994253,4.333333,3.8,4.0,3.428571,3.862319,...,4.636364,4.071429,3.777778,3.681818,4.266667,4.117647,4.444444,4.392857,4.21519,4.423077
989697609,3.826087,4.404762,4.47619,3.875,3.994253,4.333333,3.8,4.0,3.428571,3.862319,...,4.636364,4.071429,3.777778,3.681818,4.266667,4.117647,4.444444,4.392857,4.21519,4.423077


In [196]:
user_means = A_filled.mean(axis=1)

M = A_filled.sub(user_means, axis=0)

M.head()

product_id,P107306,P114902,P12045,P122651,P122661,P122718,P122727,P122762,P122767,P122774,...,P54509,P6028,P7365,P7880,P91627362,P94421,P94812,P9939,P9940,P9941
author_id,,,,,,,,,,,,,,,,,,,,,
2760276,-0.472146,0.106529,0.177957,-0.423233,-0.303980,0.035100,-0.498233,-0.298233,-0.869662,-0.435914,...,0.338131,-0.226804,-0.520455,-2.298233,-0.031566,-0.180586,0.146211,0.094624,-0.083043,0.124844
11777122,-0.474344,0.104331,0.175760,-0.425431,-0.306178,0.032902,-0.500431,-0.300431,-0.871859,-0.438112,...,0.335933,-0.229002,-0.522653,-0.618613,-0.033764,-0.182784,0.144014,0.092426,-0.085241,0.122646
967124371,-0.475407,0.103268,0.174696,-0.426494,-0.307241,0.031839,-0.501494,-0.301494,-0.872923,-0.439175,...,0.334870,-0.230065,-0.523716,-0.301494,-0.034827,-0.183847,0.142950,0.091363,-0.086304,0.121583
985770407,-0.475209,0.103466,0.174894,-0.426296,-0.307043,0.032037,-0.501296,-0.301296,-0.872725,-0.438977,...,0.335067,-0.229868,-0.523518,-0.619478,-0.034629,-0.183649,0.143148,0.091561,-0.086106,0.121781
989697609,-0.474941,0.103734,0.175162,-0.426028,-0.306775,0.032305,-0.501028,-0.301028,-0.872457,-0.438710,...,0.335335,-0.229600,-0.523251,-0.619210,-0.034362,-0.183381,0.143416,0.091829,-0.085838,0.122049


In [197]:
U, s, Vt = svd(M.values, full_matrices=False)

print("U shape:", U.shape)
print("s shape:", s.shape)
print("Vt shape:", Vt.shape)

U shape: (7029, 1752)
s shape: (1752,)
Vt shape: (1752, 1752)


In [198]:
k = 20

U_k = U[:, :k]
S_k = np.diag(s[:k])
Vt_k = Vt[:k, :]

M_hat = U_k @ S_k @ Vt_k
A_hat = M_hat + user_means.values.reshape(-1, 1)

pred_matrix = pd.DataFrame(
    A_hat,
    index=A.index,
    columns=A.columns
)

pred_matrix = pred_matrix.clip(1, 5)

pred_matrix.head()

/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_90569/4076605191.py:7: RuntimeWarning: divide by zero encountered in matmul
  M_hat = U_k @ S_k @ Vt_k
/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_90569/4076605191.py:7: RuntimeWarning: overflow encountered in matmul
  M_hat = U_k @ S_k @ Vt_k
/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_90569/4076605191.py:7: RuntimeWarning: invalid value encountered in matmul
  M_hat = U_k @ S_k @ Vt_k


product_id,P107306,P114902,P12045,P122651,P122661,P122718,P122727,P122762,P122767,P122774,...,P54509,P6028,P7365,P7880,P91627362,P94421,P94812,P9939,P9940,P9941
author_id,,,,,,,,,,,,,,,,,,,,,
2760276,3.820654,4.403768,4.477018,3.869777,3.989823,4.331923,3.794191,3.995761,3.419895,3.856922,...,4.637277,4.067843,3.771944,3.675061,4.264552,4.114306,4.443845,4.391752,4.212930,4.422366
11777122,3.825760,4.405586,4.477094,3.874782,3.994233,4.334003,3.799633,4.000038,3.427487,3.862008,...,4.637572,4.071581,3.777362,3.681221,4.267223,4.117884,4.445317,4.393622,4.215593,4.423901
967124371,3.828471,4.406421,4.476924,3.877433,3.996520,4.334982,3.802532,4.002229,3.431613,3.864697,...,4.637554,4.073514,3.780249,3.684527,4.268544,4.119723,4.445940,4.394495,4.216926,4.424585
985770407,3.826706,4.406663,4.477821,3.875788,3.995261,4.335024,3.800626,4.001080,3.428421,3.862968,...,4.638590,4.072583,3.778308,3.682206,4.268293,4.118920,4.446351,4.394666,4.216539,4.424916
989697609,3.826563,4.406287,4.477492,3.875613,3.995045,4.334686,3.800481,4.000851,3.428416,3.862817,...,4.638157,4.072342,3.778181,3.682103,4.267966,4.118656,4.445974,4.394306,4.216264,4.424551


In [199]:
true_vals = A[observed_mask].stack()
pred_vals = pred_matrix[observed_mask].stack()

rmse = np.sqrt(mean_squared_error(true_vals, pred_vals))
mae = mean_absolute_error(true_vals, pred_vals)

print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 0.7638128127210784
MAE: 0.5039713361111201


In [200]:
sample_user = A.index[0]

print("Original ratings for user:", sample_user)
A.loc[sample_user].dropna().sort_values(ascending=False)

Original ratings for user: 2760276


product_id
P232915    5.0
P420652    5.0
P309409    4.0
P386762    3.0
P446612    3.0
P448563    3.0
P427416    2.0
P7880      2.0
P444222    1.0
Name: 2760276, dtype: float64

In [201]:
print("Predicted ratings for user:", sample_user)
pred_matrix.loc[sample_user].sort_values(ascending=False).head(20)

Predicted ratings for user: 2760276


product_id
P416725    5.0
P458964    5.0
P504001    5.0
P433469    5.0
P442745    5.0
P467969    5.0
P457522    5.0
P422848    5.0
P474107    5.0
P501587    5.0
P504042    5.0
P481371    5.0
P481834    5.0
P420638    5.0
P481973    5.0
P504050    5.0
P476436    5.0
P468652    5.0
P433621    5.0
P469525    5.0
Name: 2760276, dtype: float64

In [202]:
user_compare = pd.DataFrame({
    "original_rating": A.loc[sample_user],
    "predicted_rating": pred_matrix.loc[sample_user]
})

user_compare.sort_values("predicted_rating", ascending=False).head(20)

,original_rating,predicted_rating
product_id,,
P416725,NaN,5.0
P458964,NaN,5.0
P504001,NaN,5.0
P433469,NaN,5.0
P442745,NaN,5.0
P467969,NaN,5.0
P457522,NaN,5.0
P422848,NaN,5.0
P474107,NaN,5.0


In [203]:
def recommend_for_user(user_id, A, pred_matrix, top_n=10):
    if user_id not in A.index:
        print("User not found.")
        return None
    
    already_rated = A.loc[user_id].dropna().index
    preds = pred_matrix.loc[user_id].drop(labels=already_rated)
    
    recs = preds.sort_values(ascending=False).head(top_n).reset_index()
    recs.columns = ["product_id", "predicted_rating"]
    return recs

recs = recommend_for_user(sample_user, A, pred_matrix, top_n=10)
recs

,product_id,predicted_rating
0,P469516,5.0
1,P504001,5.0
2,P422848,5.0
3,P423145,5.0
4,P423164,5.0
5,P470246,5.0
6,P482681,5.0
7,P470247,5.0
8,P482327,5.0
9,P427641,5.0


In [204]:
recs = recs.merge(
    df_products[["product_id", "product_name", "brand_name", "primary_category"]],
    on="product_id",
    how="left"
)

recs

,product_id,predicted_rating,product_name,brand_name,primary_category
0,P469516,5.0,CLEAR Ultra-Light Daily Hydrating Fluid SPF 30+,Paula's Choice,Skincare
1,P504001,5.0,Ceramidin Skin Barrier Serum Toner,Dr. Jart+,Skincare
2,P422848,5.0,Crème Riche Anti-Aging Peptide Night Cream,Tata Harper,Skincare
3,P423145,5.0,Age Reversal Eye Complex,Dermalogica,Skincare
4,P423164,5.0,Future Solution LX Extra Rich Cleansing Foam,Shiseido,Skincare
5,P470246,5.0,Calming Hydration Booster Serum,ROSE Ingleton MD,Skincare
6,P482681,5.0,Bye Bye Makeup 3-in-1 Makeup Melting Cleansing...,IT Cosmetics,Skincare
7,P470247,5.0,Blemish Control Booster Serum,ROSE Ingleton MD,Skincare
8,P482327,5.0,Mini Squalane Cleanser,The Ordinary,Skincare
9,P427641,5.0,Mini Confidence in a Cream Hydrating Moisturizer,IT Cosmetics,Skincare


In [205]:
df_mf = df_phaseA_filtered[["author_id", "product_id", "rating"]].copy()

train_df, test_df = train_test_split(df_mf, test_size=0.2, random_state=42)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (106976, 3)
Test shape: (26744, 3)


In [206]:
A_train = train_df.pivot(index="author_id", columns="product_id", values="rating")
A_train = A_train.reindex(index=A.index, columns=A.columns)

In [207]:
global_mean_train = train_df["rating"].mean()

A_train_filled = A_train.fillna(A_train.mean()).fillna(global_mean_train)

user_means_train = A_train_filled.mean(axis=1)
M_train = A_train_filled.sub(user_means_train, axis=0)

In [208]:
U, s, Vt = svd(M_train.values, full_matrices=False)

k = 20
U_k = U[:, :k]
S_k = np.diag(s[:k])
Vt_k = Vt[:k, :]

A_train_hat = (U_k @ S_k @ Vt_k) + user_means_train.values.reshape(-1, 1)

pred_train_matrix = pd.DataFrame(
    A_train_hat,
    index=A.index,
    columns=A.columns
).clip(1, 5)

/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_90569/3243762526.py:8: RuntimeWarning: divide by zero encountered in matmul
  A_train_hat = (U_k @ S_k @ Vt_k) + user_means_train.values.reshape(-1, 1)
/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_90569/3243762526.py:8: RuntimeWarning: overflow encountered in matmul
  A_train_hat = (U_k @ S_k @ Vt_k) + user_means_train.values.reshape(-1, 1)
/var/folders/7w/vrg67xp53xq_j1lvk8w5th380000gn/T/ipykernel_90569/3243762526.py:8: RuntimeWarning: invalid value encountered in matmul
  A_train_hat = (U_k @ S_k @ Vt_k) + user_means_train.values.reshape(-1, 1)


In [209]:
y_true = []
y_pred = []

for _, row in test_df.iterrows():
    u = row["author_id"]
    p = row["product_id"]
    r = row["rating"]
    
    if u in pred_train_matrix.index and p in pred_train_matrix.columns:
        y_true.append(r)
        y_pred.append(pred_train_matrix.loc[u, p])

rmse_test = np.sqrt(mean_squared_error(y_true, y_pred))
mae_test = mean_absolute_error(y_true, y_pred)

print("Test RMSE:", rmse_test)
print("Test MAE:", mae_test)

Test RMSE: 0.8396372430468843
Test MAE: 0.5800717687812766
